# Production Agent Architecture

**Level:** Advanced · **Time:** 60 min

Building an agent in a notebook is easy. Deploying it to a Kubernetes cluster where pods crash, networks drop, and queues back up is hard.

In this notebook, we will simulate two critical production patterns:
1. **Durable Execution (Checkpointing):** An agent saves its state to a database before "crashing", and successfully resumes on reboot.
2. **Idempotency Enforcement:** An agent hallucinates and tries to charge a credit card twice for the same transaction. The Tool Gateway prevents the double-charge.

---
## Pattern 1: Durable Execution (Checkpointers)

An agent proposes a Terraform deployment and needs human approval. Instead of blocking the thread (`time.sleep`), it saves its state to a simulated PostgreSQL database and exits. When the "human" approves, a new worker resumes the exact state.

In [ ]:
# Simulated PostgreSQL Checkpointer
db_checkpoints = {}

def execute_agent_step(session_id, current_state):
    print(f"\n[Worker Node 1] Starting execution for Session {session_id}...")
    
    if current_state == "INIT":
        print("[Worker Node 1] Step 1: Drafted Terraform Plan.")
        print("[Worker Node 1] Requires HITL Approval. Saving Checkpoint to DB and exiting thread.")
        # Checkpointing!
        db_checkpoints[session_id] = "AWAITING_APPROVAL"
        return
        
    if current_state == "APPROVED":
        print(f"[Worker Node 2] Step 2: Resuming from checkpoint.")
        print(f"[Worker Node 2] Executing Terraform Apply... Success!")
        db_checkpoints[session_id] = "COMPLETED"
        return

# 1. Start the job
execute_agent_step("session-123", "INIT")

# 2. Server dies, time passes. Human clicks "Approve" in the UI.
print("\n--- 3 Days Later: Human clicks 'Approve' ---")
db_checkpoints["session-123"] = "APPROVED"

# 3. A totally different worker node picks up the webhook event
execute_agent_step("session-123", db_checkpoints["session-123"])



[Worker Node 1] Starting execution for Session session-123...
[Worker Node 1] Step 1: Drafted Terraform Plan.
[Worker Node 1] Requires HITL Approval. Saving Checkpoint to DB and exiting thread.

--- 3 Days Later: Human clicks 'Approve' ---

[Worker Node 2] Starting execution for Session session-123...
[Worker Node 2] Step 2: Resuming from checkpoint.
[Worker Node 2] Executing Terraform Apply... Success!


---
## Pattern 2: Idempotency Enforcement

An LLM receives a network timeout while calling `charge_credit_card`. Assuming the call failed, the LLM retries. Because the orchestrator attached a strict `ToolCallID` (Idempotency Key), the Tool Gateway catches the duplicate and prevents a double charge.

In [ ]:
# Simulated Redis Cache for Idempotency
idempotency_cache = {}

def charge_credit_card(amount: int, tool_call_id: str) -> str:
    print(f"\n[Tool Gateway] Intercepted tool call ID: {tool_call_id}")
    
    # Check Idempotency Cache
    if tool_call_id in idempotency_cache:
        print("[Tool Gateway] CACHE HIT! Duplicate UUID detected. Returning cached response to prevent double-charge.")
        return idempotency_cache[tool_call_id]
        
    # Execute actual business logic
    print(f"[Banking API] Charging card for ${amount}...")
    success_response = "200 OK: Card Charged Successfully."
    
    # Save to cache
    idempotency_cache[tool_call_id] = success_response
    return success_response


# Attempt 1: The agent calls the tool. It succeeds, but the network drops before the agent sees the response.
print("[Agent] Calling charge_credit_card...")
charge_credit_card(amount=50, tool_call_id="call_abc123")
print("[Agent] Network Timeout! I didn't get a response. I will retry.")

# Attempt 2: The agent retries with the SAME tool call ID.
print("\n[Agent] RETRYING charge_credit_card...")
result = charge_credit_card(amount=50, tool_call_id="call_abc123")
print(f"[Agent] Received Response: {result}")


[Agent] Calling charge_credit_card...

[Tool Gateway] Intercepted tool call ID: call_abc123
[Banking API] Charging card for $50...
[Agent] Network Timeout! I didn't get a response. I will retry.

[Agent] RETRYING charge_credit_card...

[Tool Gateway] Intercepted tool call ID: call_abc123
[Tool Gateway] CACHE HIT! Duplicate UUID detected. Returning cached response to prevent double-charge.
[Agent] Received Response: 200 OK: Card Charged Successfully.
